In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tkinter import messagebox
from tkinter import *
from tkinter.filedialog import askopenfilename
from tkinter import simpledialog
import tkinter
from tkinter import filedialog
import os
from sklearn.model_selection import train_test_split 
from sklearn import metrics
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score 
from sklearn import svm
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import VotingClassifier
import socket
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import xgboost as xgb


In [2]:
dataset = pd.read_csv('dataset/new_data.csv')
dataset

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,1,89,66,23,94,28.1,0.167,21,0
1,0,137,40,35,168,43.1,2.288,33,1
2,8,183,64,0,0,23.3,0.672,32,1
3,0,137,40,35,168,43.1,2.288,33,1
4,0,137,40,35,168,43.1,2.288,33,1
...,...,...,...,...,...,...,...,...,...
9995,1,85,66,29,0,26.6,0.351,31,0
9996,8,183,64,0,0,23.3,0.672,32,1
9997,1,85,66,29,0,26.6,0.351,31,0
9998,1,89,66,23,94,28.1,0.167,21,0


In [3]:
dataset.shape

(10000, 9)

In [4]:
dataset['Outcome'].value_counts()

1    6009
0    3991
Name: Outcome, dtype: int64

In [5]:
y = dataset['Outcome']
X = dataset.drop(['Outcome'], axis = 1)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [7]:
lgbm = lgb.LGBMClassifier(
    n_estimators=50, learning_rate=0.3, max_depth=52, num_leaves=20, 
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=1
)

rf = RandomForestClassifier(
    n_estimators=50, max_depth=55, min_samples_split=50, min_samples_leaf=10, 
    bootstrap=True, random_state=42, n_jobs=1
)

xgb_model = xgb.XGBClassifier(
    n_estimators=50, learning_rate=0.3, max_depth=52, gamma=0.1, 
    reg_alpha=0.01, colsample_bytree=0.8, subsample=0.8, 
    use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=1
)

# Ensemble model using soft voting
estimators = [
    ('lgbm', lgbm),
    ('rf', rf),
    ('xgb', xgb_model)
]

ensemble = VotingClassifier(estimators, voting='hard', n_jobs=1)  # Soft voting improves performance
ensemble.fit(X_train, y_train)  # Train the ensemble model
y_pred = ensemble.predict(X_test)  # Make predictions

# Improved accuracy calculation
ensemble_acc = accuracy_score(y_test, y_pred) * 100  

print("Ensemble Accuracy : {:.2f}%\n".format(ensemble_acc))

Ensemble Accuracy : 100.00%



In [ ]:
for name, model in [('LGBM', lgbm), ('Random Forest', rf), ('XGBoost', xgb_model)]:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    print(f"\n===== {name} =====")
    print("Accuracy :", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall   :", recall_score(y_test, y_pred))
    print("F1 Score :", f1_score(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=model.classes_)
    disp.plot(cmap=plt.cm.Blues, values_format='d')
    plt.title(f"Confusion Matrix - {name}")
    plt.show()

    # ROC Curve
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    auc_score = roc_auc_score(y_test, y_pred_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {auc_score:.2f})')

plt.plot([0, 1], [0, 1], 'k--')
plt.title('ROC Curves for All Models')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.legend()
plt.grid()
plt.show()
